# MLPerf Vision Benchmark
Runs MLPerf Inference benchmarks for image classification using the **corespan/inference** repo.
Uses the official `run_local.sh` → `python/main.py` pipeline.
`mlperf_loadgen` is pre-built in this image — no compilation needed.

**Run cells top to bottom. Edit only Cell 2.**

## Cell 1 — GPU Verification

In [ ]:
import torch
print('=== GPU Status ===')
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i} : {p.name}  {p.total_memory/1e9:.1f} GB VRAM')
    a = torch.rand(1024,1024,device='cuda') @ torch.rand(1024,1024,device='cuda')
    torch.cuda.synchronize()
    print('Smoke test : PASSED')
else:
    print('WARNING: No GPU — benchmark runs on CPU')


## Cell 2 — Configuration
Edit this cell to select the model and run settings.

In [ ]:
# ── Edit these ──────────────────────────────────────────────────────────────
MODEL_NAME  = 'resnet50'       # Options: 'resnet50', 'resnet18'
BACKEND     = 'onnxruntime'    # onnxruntime or pytorch
SCENARIO    = 'SingleStream'   # SingleStream | Offline | MultiStream
TIME_LIMIT  = 30               # seconds (set None for full official run)
RUN_ACCURACY = True

# ── NFS paths ────────────────────────────────────────────────────────────────
NFS_MODEL_DIR = '/data/benchmarking/models'     # resnet50.pth, resnet18.pth
NFS_IMAGE_DIR = '/data/ResnetInference/images'  # ImageNet-style images

# ── Benchmark repo ───────────────────────────────────────────────────────────
INFERENCE_DIR = '/tmp/corespan_inference'
VISION_DIR    = f'{INFERENCE_DIR}/vision/classification_and_detection'
ONNX_PATH     = f'/tmp/{MODEL_NAME}.onnx'

print(f'Model    : {MODEL_NAME}')
print(f'Backend  : {BACKEND}')
print(f'Scenario : {SCENARIO}')
print(f'Time     : {TIME_LIMIT}s')


## Cell 3 — Clone corespan/inference (vision only, sparse checkout)

In [ ]:
import os, subprocess

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if out: print(out[-2000:])
    if r.returncode != 0: raise RuntimeError(f'Failed: {cmd}')

if os.path.isdir(VISION_DIR):
    print(f'Already cloned: {VISION_DIR}')
else:
    print('Sparse-cloning vision/classification_and_detection ...')
    run('git clone --depth 1 --filter=blob:none --sparse https://github.com/corespan/inference.git ' + INFERENCE_DIR)
    run(f'cd {INFERENCE_DIR} && git sparse-checkout set vision/classification_and_detection')
    run(f'cd {VISION_DIR} && pip install -q -e .')
    print('Done.')


## Cell 4 — Convert .pth → ONNX
MLPerf vision uses onnxruntime. The NFS has `.pth` (PyTorch) files.
Cached in `/tmp` — skipped on re-runs.

In [ ]:
import os, torch, torchvision

pth = f'{NFS_MODEL_DIR}/{MODEL_NAME}.pth'
assert os.path.exists(pth), f'Not found on NFS: {pth}'

if os.path.exists(ONNX_PATH):
    print(f'Cache hit: {ONNX_PATH}')
else:
    print(f'Converting {pth} -> {ONNX_PATH} ...')
    model = getattr(torchvision.models, MODEL_NAME)(weights=None)
    ckpt  = torch.load(pth, map_location='cpu')
    model.load_state_dict(ckpt['state_dict'] if 'state_dict' in ckpt else ckpt)
    model.eval()
    dummy = torch.zeros(1, 3, 224, 224)
    torch.onnx.export(model, dummy, ONNX_PATH,
        input_names=['input'], output_names=['output'],
        dynamic_axes={'input':{0:'batch'},'output':{0:'batch'}},
        opset_version=17)
    print(f'Saved: {ONNX_PATH}')


## Cell 5 — Prepare Dataset
Uses NFS images if available, falls back to fake ImageNet for quick tests.

In [ ]:
import os, subprocess

if os.path.isdir(NFS_IMAGE_DIR) and len(os.listdir(NFS_IMAGE_DIR)) > 0:
    DATA_DIR = NFS_IMAGE_DIR
    print(f'NFS images : {DATA_DIR}  ({len(os.listdir(DATA_DIR))} files)')
else:
    DATA_DIR = '/tmp/fake_imagenet'
    if not os.path.isdir(DATA_DIR):
        print('Generating fake ImageNet ...')
        r = subprocess.run(f'cd {VISION_DIR} && bash tools/make_fake_imagenet.sh {DATA_DIR}',
            shell=True, capture_output=True, text=True)
        print(r.stdout or r.stderr or 'done')
    print(f'Fake dataset : {DATA_DIR}')

print(f'DATA_DIR = {DATA_DIR}')


## Cell 6 — Run MLPerf Benchmark
Calls `run_local.sh` from the cloned corespan/inference repo.

In [ ]:
import os, subprocess, time, shutil

# run_local.sh looks for MODEL_DIR and DATA_DIR env vars
# and expects the model file named as <model>.onnx inside MODEL_DIR
model_dir = os.path.dirname(ONNX_PATH)
link = f'{model_dir}/{MODEL_NAME}.onnx'
if not os.path.exists(link):
    shutil.copy(ONNX_PATH, link)

os.environ['MODEL_DIR'] = model_dir
os.environ['DATA_DIR']  = DATA_DIR

extra = []
if TIME_LIMIT: extra += [f'--time {TIME_LIMIT}', '--max-latency 0.5']
if RUN_ACCURACY: extra.append('--accuracy')
os.environ['EXTRA_OPS'] = ' '.join(extra)

device = 'gpu' if torch.cuda.is_available() else 'cpu'
cmd = (f'cd {VISION_DIR} && '
       f'./run_local.sh {BACKEND} {MODEL_NAME} {device} '
       f'--scenario {SCENARIO}')
if RUN_ACCURACY: cmd += ' --accuracy'

print(f'Running:\n  {cmd}\n')
t0 = time.time()
r  = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0: print('STDERR:', r.stderr[-2000:])
print(f'Wall time: {time.time()-t0:.1f}s')


## Cell 7 — Results

In [ ]:
import json, os

out_dir  = f'{VISION_DIR}/output/{MODEL_NAME}-{BACKEND}-gpu'
summary  = f'{out_dir}/mlperf_log_summary.txt'
results  = f'{out_dir}/results.json'

print('=== MLPerf Log Summary ===')
if os.path.exists(summary):
    print(open(summary).read())
else:
    print('Not found — check Cell 6 output.')

print('=== results.json ===')
if os.path.exists(results):
    for k, v in json.load(open(results)).items():
        print(f'  {k}: {v}')
else:
    print('Not found.')
